# TrailKeeper — Step 1: Setup & sanity checks

In this step we will:

* Install dependencies (into the current Jupyter kernel)
* (Optionally) log in to Weights & Biases
* Verify configs and scripts exist

In [ ]:
REPO_URL = "https://github.com/MauricioSight/TrailKeeper.git"
REPO_DIR = "TrailKeeper"

import pathlib

if not pathlib.Path(REPO_DIR).exists():
    print("Cloning TrailKeeper...")
    !git clone $REPO_URL

# enter repo
%cd $REPO_DIR
print("Python version in this kernel:")
!python -V

Cloning TrailKeeper...
c:\Projects\26.2\TADI\intrusion_detection_studies\TrailKeeper
Python version in this kernel:


Cloning into 'TrailKeeper'...


Python 3.13.7


In [2]:
print("Installing dependencies (this uses the current Jupyter kernel)...")
!pip install -r requirements.txt

Installing dependencies (this uses the current Jupyter kernel)...
^C


### (Optional) Weights & Biases login

If you want experiment tracking dashboards, log in to W&B. You can either:

* run `!wandb login` and paste your API key, **or**
* set the environment variable `WANDB_API_KEY` beforehand.

In [8]:
!wandb login --relogin

wandb: ERROR Find detailed error logs at: C:\Users\gabri\AppData\Local\Temp\debug-cli.gabri.log
Error: No API key configured. Use `wandb login` to log in.


In [6]:
import os

if os.environ.get("WANDB_API_KEY"):
    print("WANDB_API_KEY detected. You're good to go")
else:
    print("No WANDB_API_KEY found.")
    print("If you want tracking, run this and follow the prompt:")
    print("    !wandb login")

No WANDB_API_KEY found.
If you want tracking, run this and follow the prompt:
    !wandb login


### Quick sanity checks (configs & scripts)

In [7]:
import glob, os, json, textwrap, pathlib

print("Configs available:")
for p in sorted(glob.glob("configs/*.yaml")):
    print(" -", p)

print("\nTop-level scripts:")
for p in sorted(glob.glob("execute_*.py")):
    print(" -", p)

# Show the MLP config
print("\nPreview configs/mlp.yaml:")
!sed -n '1,120p' configs/mlp.yaml

Configs available:
 - configs\mlp.yaml
 - configs\tune-mlp.yaml

Top-level scripts:
 - execute_get_metrics.py
 - execute_test.py
 - execute_train_validation.py
 - execute_tunning.py

Preview configs/mlp.yaml:


'sed' nao � reconhecido como um comando interno
ou externo, um programa oper�vel ou um arquivo em lotes.


# Step 2: First Experiment

Now let’s run our **first training + validation** with the provided MLP config on MNIST.

What you’ll see:

* A new folder in `runs/` with timestamped name
* Saved **config**, **metrics**, **logs**, **predictions**, and **model weights**
* (Optional) live tracking in [W&B](https://wandb.ai/) if enabled

In [9]:
!python execute_train_validation.py --config mlp

Runs directory not found at runs


2026-10-29 16:03:14,211 | INFO | Initiating training and validation...
2026-10-29 16:03:14,211 | INFO | [ RUN ID: mlp_MNIST_20261029_160314 ]

  0%|          | 0.00/9.91M [00:00<?, ?B/s]
  0%|          | 32.8k/9.91M [00:00<00:59, 166kB/s]
  1%|          | 98.3k/9.91M [00:00<00:41, 238kB/s]
  2%|▏         | 164k/9.91M [00:00<00:34, 283kB/s] 
  3%|▎         | 262k/9.91M [00:00<00:26, 363kB/s]
  4%|▍         | 426k/9.91M [00:00<00:15, 605kB/s]
  6%|▋         | 623k/9.91M [00:01<00:10, 896kB/s]
  9%|▉         | 885k/9.91M [00:01<00:07, 1.26MB/s]
 11%|█         | 1.08M/9.91M [00:01<00:06, 1.27MB/s]
 14%|█▍        | 1.38M/9.91M [00:01<00:05, 1.54MB/s]
 17%|█▋        | 1.64M/9.91M [00:01<00:05, 1.53MB/s]
 19%|█▉        | 1.87M/9.91M [00:01<00:04, 1.64MB/s]
 21%|██        | 2.06M/9.91M [00:02<00:14, 533kB/s] 
 35%|███▌      | 3.47M/9.91M [00:02<00:03, 1.70MB/s]
 39%|███▊      | 3.83M/9.91M [00:03<00:03, 1.64MB/s]
 42%|████▏     | 4.13M/9.91M [00:03<00:03, 1.69MB/s]
 44%|████▍     | 4.39M/9.91M

Every experiment is stored in the `runs/` folder.
Let’s peek at what was just created.

In [10]:
import os

runs = sorted(os.listdir("runs"))
print("Available runs:", runs[-3:])  # last few runs
last_run = os.path.join("runs", runs[-1])
print("Latest run folder:", last_run)

print("\nFiles inside the run:")
os.listdir(last_run)

Available runs: ['mlp_MNIST_20261029_160314']
Latest run folder: runs\mlp_MNIST_20261029_160314

Files inside the run:


['config.yaml',
 'model.pt',
 'train_confusion_matrix.png',
 'train_labels_predictions.pt',
 'train_metrics.json',
 'train_output.log']

In [11]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

### Look inside artifacts

* `config.yaml`: snapshot of hyperparameters
* `train_metrics.json` or `.csv`: evaluation results
* `train_output.*`: model raw outputs
* `model.pt` (or similar): saved weights

Let’s preview the metrics file.

In [12]:
import json, glob

metrics_files = glob.glob(os.path.join(last_run, "train_metrics*"))
print("Metrics files:", metrics_files)

if metrics_files:
    with open(metrics_files[0]) as f:
        print(json.dumps(json.load(f), indent=2))

Metrics files: ['runs\\mlp_MNIST_20261029_160314\\train_metrics.json']
{
  "acc": 0.9613333333333334,
  "f1_macro": 0.9609365090295532,
  "f1_weighted": 0.9612787248718666,
  "bal_acc": 0.960318571064868,
  "top5": 0.9978333333333333,
  "ova_auc": 0.9981940863264505,
  "ce": 0.16709832847118378
}


# Step 3: recompute metrics (no retraining)

every run saves **predictions** + **true labels**, so you can recompute metrics later on.
this is essential for:

* reproducibility
* trying new evaluation criteria
* fair comparisons across experiments

In [13]:
last_run = runs[-1]

In [14]:
!python execute_get_metrics.py --run_id $last_run

2026-10-29 16:05:05,100 | INFO | Initiating get metrics...
2026-10-29 16:05:05,100 | INFO | [ RUN ID: mlp_MNIST_20261029_160314 ]
2026-10-29 16:05:05,192 | INFO | Data loaded successfully.
2026-10-29 16:05:08,273 | INFO | 
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1127
           1       0.98      0.99      0.98      1345
           2       0.96      0.97      0.96      1241
           3       0.95      0.95      0.95      1182
           4       0.96      0.96      0.96      1200
           5       0.95      0.95      0.95      1104
           6       0.98      0.97      0.97      1201
           7       0.93      0.98      0.95      1336
           8       0.96      0.95      0.96      1154
           9       0.97      0.91      0.94      1110

    accuracy                           0.96     12000
   macro avg       0.96      0.96      0.96     12000
weighted avg       0.96      0.96      0.96     12000

2026-10-29 16:05:0

# Step 4: Manual hyperparameter changes

You can **manually edit the config** to try different hyperparameters.

open `configs/mlp.yaml` and experiment with values like:

```yaml
modeling:
  structure:
    num_layers: 16 # 👈 try fewer or more layers

  training:
    learning_rate: 0.0001  # 👈 try changing the lr
```

In [15]:
!python execute_train_validation.py --config mlp

2026-10-29 16:07:23,902 | INFO | Initiating training and validation...
2026-10-29 16:07:23,902 | INFO | [ RUN ID: mlp_MNIST_20261029_160723 ]
2026-10-29 16:07:27,059 | INFO | Loading cached data from: data/MNIST/processed/train_norm.pt
2026-10-29 16:07:27,267 | INFO | Data loaded successfully.
2026-10-29 16:07:27,268 | INFO | Using device: cpu
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\gabri\_netrc.
wandb: Currently logged in as: marquesgbr (marquesgbr-ufpe) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin
wandb: setting up run bzuq2vrf
wandb: Tracking run with wandb version 0.31.0
wandb: Run data is saved locally in c:\Projects\26.2\TADI\intrusion_detection_studies\TrailKeeper\wandb\run-20261029_160731-bzuq2vrf
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run mlp_MNIST_20261029_160723
wandb:  View project at https://forge.coreweave.com/wandb/marquesgbr-ufpe/MINIST
wandb:  View run at htt

### compare results

1. check the new run folder in `runs/`
2. compare metrics with your previous run (accuracy, loss, etc.)
3. ask: *how did changing `lr` or `epochs` affect the outcome?*

In [16]:
import os, json, glob

runs = sorted(os.listdir("runs"))
last_run = os.path.join("runs", runs[-1])
metrics_files = glob.glob(os.path.join(last_run, "train_metrics*"))

if metrics_files:
    with open(metrics_files[0]) as f:
        metrics = json.load(f)
    print("Metrics from last run:")
    print(json.dumps(metrics, indent=2))

Metrics from last run:
{
  "acc": 0.8898333333333334,
  "f1_macro": 0.8875219149339808,
  "f1_weighted": 0.8896949642035036,
  "bal_acc": 0.8874055480779616,
  "top5": 0.9799166666666667,
  "ova_auc": 0.9888722211691393,
  "ce": 0.38942137360572815
}


# step 5: hyperparameter tuning with optuna

trailkeeper integrates with **optuna** to automate hyperparameter search.
you define a **search space** in a config file (e.g., `configs/tune-mlp.yaml`), and trailkeeper will:

* sample hyperparameters
* train models
* log metrics
* record the best trial

In [25]:
!python execute_tunning.py --config tune-mlp

2026-10-15 13:59:07,311 | INFO | Starting tuning process...
2026-10-15 13:59:07,311 | INFO | [ RUN ID: optuna_tpe_mlp_20261015_135907 ]
2026-10-15 13:59:07,311 | INFO | Initializing optimizer...
[I 2026-10-15 13:59:08,714] A new study created in RDB with name: optuna_tpe_mlp_20261015_135907
2026-10-15 13:59:08,838 | INFO | Trial parameters: 
2026-10-15 13:59:08,838 | INFO | {
    "modeling-structure-num_layers": 5,
    "modeling-structure-hidden_size": 65,
    "modeling-training-num_epochs": 9,
    "modeling-training-learning_rate": 1e-05,
    "modeling-training-batch_size": 64
}
2026-10-15 13:59:08,853 | INFO | Initiating training and validation...
2026-10-15 13:59:08,853 | INFO | [ RUN ID: mlp_MNIST_20261015_135908 ]
2026-10-15 13:59:10,623 | INFO | Loading cached data from: data/MNIST/processed/train_norm.pt
2026-10-15 13:59:10,810 | INFO | Data loaded successfully.
2026-10-15 13:59:10,812 | INFO | Using device: cpu
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai 

### inspect results

after tuning:

* check the `runs/` folder for multiple trial runs
* see the best parameters and score in the logs (and in W&B if enabled)

In [ ]:
import os

runs = sorted(os.listdir("runs"))
print("Last few runs after tuning:", runs[-5:])

Last few runs after tuning: ['mlp_MNIST_20260514_122020', 'mlp_MNIST_20260514_122045', 'mlp_MNIST_20260514_122129', 'mlp_MNIST_20260514_122205', 'optuna_tpe_mlp_20260514_121948']


# Step 6: Add a CNN model

In this step, we will integrate a **Convolutional Neural Network (CNN)** into TrailKeeper, using the changes from the PR. [GitHub](https://github.com/MauricioSight/TrailKeeper/pull/1/files)

* Inspect and understand the CNN implementation
* Register it in the model factory
* Add a config file for CNN
* Run training + evaluation with CNN
* Compare results with the MLP baseline

#### What changed / where to hook in

From the PR:

* **`configs/cnn.yaml`** was added, defining structure (input dimensions, kernel size, number of layers, channels, dropout, hidden size, output size) and training hyperparameters. [GitHub](https://github.com/MauricioSight/TrailKeeper/pull/1/files)
* **`modeling/structure/cnn.py`** contains the new `CNN` class extending `PytorchModelStructure`. It constructs the convolutional layers, pooling, flatten, linear layers, etc. [GitHub](https://github.com/MauricioSight/TrailKeeper/pull/1/files)
* **`modeling/structure/factory.py`** was updated: when `name == 'cnn'`, it now returns `CNN`. [GitHub](https://github.com/MauricioSight/TrailKeeper/pull/1/files)

So to integrate the CNN, you need:

1. The config file in `configs/` for `"cnn"`.
2. CNN class in `modeling/structure/`.
3. Factory mapping from `"cnn"` to the class.
4. (Optionally) In the YAML, reference preprocessing, metrics, etc., consistent with other models.

In [20]:
!python execute_train_validation.py --config cnn

2026-10-29 16:14:16,122 | INFO | Initiating training and validation...
2026-10-29 16:14:16,123 | INFO | [ RUN ID: cnn_MNIST_20261029_161416 ]
2026-10-29 16:14:19,432 | INFO | Loading cached data from: data/MNIST/processed/train_norm.pt
2026-10-29 16:14:19,571 | INFO | Data loaded successfully.
2026-10-29 16:14:19,572 | INFO | Using device: cpu
wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\gabri\_netrc.
wandb: Currently logged in as: marquesgbr (marquesgbr-ufpe) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin
wandb: setting up run 25fdrxef
wandb: Tracking run with wandb version 0.31.0
wandb: Run data is saved locally in c:\Projects\26.2\TADI\intrusion_detection_studies\TrailKeeper\wandb\run-20261029_161424-25fdrxef
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run cnn_MNIST_20261029_161416
wandb:  View project at https://forge.coreweave.com/wandb/marquesgbr-ufpe/intrusion_detection_studies
wa

### Inspect the CNN run & compare

* Look at the generated `runs/` folder.
* Open `metrics.json` / CSV of that run and compare with your MLP runs.
* Think: how does CNN perform compared to MLP? Where does it help (e.g. on image data)?

In [ ]:
import os, glob, json

runs = sorted(os.listdir("runs"))
latest = os.path.join("runs", runs[-1])
print("Latest run:", latest)

mf = glob.glob(os.path.join(latest, "metrics*"))
if mf:
    with open(mf[0]) as f:
        print("Metrics:", json.dumps(json.load(f), indent=2))

Latest run: runs\mlp_MNIST_20261029_160723


[MLP](https://forge.coreweave.com/wandb/marquesgbr-ufpe/MINIST):

    - test loss: 0.17042
    - test f1-weighted: 0.96128
    - test ova_auc: 0.99819
    GPU performance and Memory cost:
        - power usage: 17%
        - enforced power limit: 35W
        - GPU temperature: 41C
        - process memory in use: 781MB

[CNN](https://forge.coreweave.com/wandb/marquesgbr-ufpe/intrusion_detection_studies?):
    
    - test loss: 1.46988
    - test f1-weighted: 0.98991
    - test ova_auc: 0.99906
    GPU performance and Memory cost:
        - power usage: 19%
        - enforced power limit: 35W
        - GPU temperature: 41C
        - process memory in use: 846MB

# Final step (exercise): add your own model

1. Make changes to the CNN and check if it achieves better results.
2. Find the best hyperparameters for both the CNN and the MLP models.
3. Analyze which model performs best and try to reason why.